# Phase 4: Maintenance Effectiveness Evaluation

This notebook evaluates the effectiveness of preventive vs corrective maintenance by:
- Comparing maintenance outcomes and costs
- Calculating ROI for each maintenance type
- Identifying optimal preventive maintenance schedules
- Analyzing effectiveness by equipment family
- Providing actionable recommendations

**Objective**: Directly addresses the requirement "Évaluer l'efficacité des deux types de maintenance"

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import pickle

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"Notebook executed on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python libraries loaded successfully")

Notebook executed on: 2026-01-26 16:19:21
Python libraries loaded successfully


In [2]:
# Define paths
data_dir = Path('data')
features_dir = data_dir / 'features'
models_dir = data_dir / 'models'
output_dir = Path('analysis_outputs')

# Create directories if needed
models_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {Path.cwd()}")
print(f"Data directory: {data_dir}")
print(f"Models will be saved to: {models_dir}")

Working directory: c:\Users\walid\OneDrive\Desktop\gmao-ai-project
Data directory: data
Models will be saved to: data\models


In [3]:
# Load all necessary datasets
print("Loading datasets...")

corrective = pd.read_csv(data_dir / 'corrective_integrated.csv')
preventive = pd.read_csv(data_dir / 'preventive_cleaned.csv')
spare_parts = pd.read_csv(data_dir / 'spare_parts_cleaned.csv')
features = pd.read_csv(features_dir / 'corrective_features.csv')
asset_master = pd.read_csv(data_dir / 'asset_master.csv')

# Convert datetime columns with mixed format handling
date_cols_corrective = ['date_creation_ot', 'date_cloture', 'date_declaration', 'date_debut_reparation']
for col in date_cols_corrective:
    if col in corrective.columns:
        corrective[col] = pd.to_datetime(corrective[col], format='mixed', errors='coerce')

date_cols_preventive = ['date_creation_ot', 'date_cloture']
for col in date_cols_preventive:
    if col in preventive.columns:
        preventive[col] = pd.to_datetime(preventive[col], format='mixed', errors='coerce')

print(f"\nDatasets loaded:")
print(f"- Corrective maintenance: {corrective.shape}")
print(f"- Preventive maintenance: {preventive.shape}")
print(f"- Spare parts: {spare_parts.shape}")
print(f"- Features: {features.shape}")
print(f"- Asset master: {asset_master.shape}")

Loading datasets...

Datasets loaded:
- Corrective maintenance: (2035, 59)
- Preventive maintenance: (9149, 42)
- Spare parts: (12573, 25)
- Features: (2035, 95)
- Asset master: (7090, 11)


## 2. Calculate Key Performance Metrics

In [4]:
# Calculate duration for both maintenance types
corrective['duration_hours'] = (corrective['date_cloture'] - corrective['date_creation_ot']).dt.total_seconds() / 3600
preventive['duration_hours'] = (preventive['date_cloture'] - preventive['date_creation_ot']).dt.total_seconds() / 3600

# Remove invalid durations
corrective = corrective[corrective['duration_hours'] > 0].copy()
preventive = preventive[preventive['duration_hours'] > 0].copy()

print("Duration metrics calculated")
print(f"\nCorrective maintenance duration statistics:")
print(corrective['duration_hours'].describe())
print(f"\nPreventive maintenance duration statistics:")
print(preventive['duration_hours'].describe())

KeyError: 'date_creation_ot'

In [ ]:
# Calculate cost metrics
# Corrective: sum of spare parts costs per OT
corrective_costs = spare_parts.groupby('code_ot').agg({
    'prix_total': 'sum',
    'quantite': 'sum'
}).reset_index()
corrective_costs.columns = ['code_ot', 'total_parts_cost', 'total_parts_quantity']

corrective = corrective.merge(corrective_costs, on='code_ot', how='left')
corrective['total_parts_cost'] = corrective['total_parts_cost'].fillna(0)
corrective['total_parts_quantity'] = corrective['total_parts_quantity'].fillna(0)

# Estimate labor cost (assuming hourly rate)
HOURLY_LABOR_RATE = 50  # EUR per hour (adjust based on actual rates)
corrective['labor_cost'] = corrective['duration_hours'] * HOURLY_LABOR_RATE
preventive['labor_cost'] = preventive['duration_hours'] * HOURLY_LABOR_RATE

# Total cost = labor + parts
corrective['total_cost'] = corrective['labor_cost'] + corrective['total_parts_cost']
preventive['total_cost'] = preventive['labor_cost']  # Preventive typically has minimal parts cost

print(f"\nCost metrics calculated (hourly rate: {HOURLY_LABOR_RATE} EUR)")
print(f"\nCorrective maintenance costs:")
print(f"- Total labor cost: {corrective['labor_cost'].sum():,.2f} EUR")
print(f"- Total parts cost: {corrective['total_parts_cost'].sum():,.2f} EUR")
print(f"- Total cost: {corrective['total_cost'].sum():,.2f} EUR")
print(f"- Average cost per intervention: {corrective['total_cost'].mean():,.2f} EUR")

print(f"\nPreventive maintenance costs:")
print(f"- Total cost: {preventive['total_cost'].sum():,.2f} EUR")
print(f"- Average cost per intervention: {preventive['total_cost'].mean():,.2f} EUR")

## 3. Maintenance Type Comparison

In [ ]:
# Create comparison dataframe
comparison_metrics = pd.DataFrame({
    'Metric': [
        'Total Interventions',
        'Avg Duration (hours)',
        'Avg Cost (EUR)',
        'Total Cost (EUR)',
        'Avg Parts Cost (EUR)',
        'Interventions per Equipment'
    ],
    'Corrective': [
        len(corrective),
        corrective['duration_hours'].mean(),
        corrective['total_cost'].mean(),
        corrective['total_cost'].sum(),
        corrective['total_parts_cost'].mean(),
        len(corrective) / corrective['code_equipement'].nunique()
    ],
    'Preventive': [
        len(preventive),
        preventive['duration_hours'].mean(),
        preventive['total_cost'].mean(),
        preventive['total_cost'].sum(),
        0,  # Preventive typically has no parts
        len(preventive) / preventive['code_equipement'].nunique()
    ]
})

comparison_metrics['Difference (%)'] = (
    (comparison_metrics['Corrective'] - comparison_metrics['Preventive']) / 
    comparison_metrics['Preventive'] * 100
)

print("\nMaintenance Type Comparison:")
print(comparison_metrics.to_string(index=False))

# Save comparison
comparison_metrics.to_csv(output_dir / 'maintenance_comparison.csv', index=False)
print(f"\nComparison saved to: {output_dir / 'maintenance_comparison.csv'}")

In [ ]:
# Visualize comparison
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Average Duration (hours)', 'Average Cost (EUR)', 
                    'Total Interventions', 'Interventions per Equipment'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# Duration comparison
fig.add_trace(
    go.Bar(x=['Corrective', 'Preventive'], 
           y=[corrective['duration_hours'].mean(), preventive['duration_hours'].mean()],
           name='Duration',
           marker_color=['#FF6B6B', '#4ECDC4']),
    row=1, col=1
)

# Cost comparison
fig.add_trace(
    go.Bar(x=['Corrective', 'Preventive'], 
           y=[corrective['total_cost'].mean(), preventive['total_cost'].mean()],
           name='Cost',
           marker_color=['#FF6B6B', '#4ECDC4']),
    row=1, col=2
)

# Total interventions
fig.add_trace(
    go.Bar(x=['Corrective', 'Preventive'], 
           y=[len(corrective), len(preventive)],
           name='Interventions',
           marker_color=['#FF6B6B', '#4ECDC4']),
    row=2, col=1
)

# Interventions per equipment
fig.add_trace(
    go.Bar(x=['Corrective', 'Preventive'], 
           y=[len(corrective) / corrective['code_equipement'].nunique(),
              len(preventive) / preventive['code_equipement'].nunique()],
           name='Rate',
           marker_color=['#FF6B6B', '#4ECDC4']),
    row=2, col=2
)

fig.update_layout(height=800, showlegend=False, title_text="Maintenance Type Comparison")
fig.show()

fig.write_html(output_dir / 'maintenance_comparison.html')
print(f"Visualization saved to: {output_dir / 'maintenance_comparison.html'}")

## 4. Equipment Family Analysis

In [ ]:
# Analyze by equipment family
corrective_by_family = corrective.groupby('famille_equipement').agg({
    'code_ot': 'count',
    'duration_hours': 'mean',
    'total_cost': ['mean', 'sum'],
    'total_parts_cost': 'mean'
}).round(2)

corrective_by_family.columns = ['_'.join(col).strip() for col in corrective_by_family.columns.values]
corrective_by_family = corrective_by_family.reset_index()
corrective_by_family.columns = ['famille_equipement', 'n_interventions', 'avg_duration', 
                                 'avg_cost', 'total_cost', 'avg_parts_cost']

preventive_by_family = preventive.groupby('famille_equipement').agg({
    'code_ot': 'count',
    'duration_hours': 'mean',
    'total_cost': ['mean', 'sum']
}).round(2)

preventive_by_family.columns = ['_'.join(col).strip() for col in preventive_by_family.columns.values]
preventive_by_family = preventive_by_family.reset_index()
preventive_by_family.columns = ['famille_equipement', 'n_interventions', 'avg_duration', 
                                'avg_cost', 'total_cost']

print("\nCorrective Maintenance by Equipment Family:")
print(corrective_by_family.sort_values('total_cost', ascending=False).to_string(index=False))

print("\nPreventive Maintenance by Equipment Family:")
print(preventive_by_family.sort_values('total_cost', ascending=False).to_string(index=False))

In [ ]:
# Merge for comparison
family_comparison = corrective_by_family.merge(
    preventive_by_family, 
    on='famille_equipement', 
    suffixes=('_corrective', '_preventive')
)

# Calculate effectiveness ratio (lower is better)
family_comparison['cost_ratio'] = (
    family_comparison['avg_cost_corrective'] / family_comparison['avg_cost_preventive']
)

family_comparison['intervention_ratio'] = (
    family_comparison['n_interventions_corrective'] / family_comparison['n_interventions_preventive']
)

# Calculate effectiveness score (0-100, higher is better)
# Score based on: lower corrective costs, fewer corrective interventions
family_comparison['effectiveness_score'] = 100 / (
    1 + family_comparison['cost_ratio'] + family_comparison['intervention_ratio']
) * 2

family_comparison = family_comparison.sort_values('effectiveness_score', ascending=False)

print("\nPreventive Maintenance Effectiveness by Equipment Family:")
print(family_comparison[['famille_equipement', 'cost_ratio', 'intervention_ratio', 
                         'effectiveness_score']].to_string(index=False))

family_comparison.to_csv(output_dir / 'family_effectiveness.csv', index=False)
print(f"\nFamily effectiveness saved to: {output_dir / 'family_effectiveness.csv'}")

In [ ]:
# Visualize effectiveness by family
fig = px.bar(
    family_comparison.sort_values('effectiveness_score', ascending=True),
    x='effectiveness_score',
    y='famille_equipement',
    orientation='h',
    title='Preventive Maintenance Effectiveness Score by Equipment Family',
    labels={'effectiveness_score': 'Effectiveness Score (0-100)', 
            'famille_equipement': 'Equipment Family'},
    color='effectiveness_score',
    color_continuous_scale='RdYlGn'
)

fig.update_layout(height=600)
fig.show()

fig.write_html(output_dir / 'family_effectiveness.html')
print(f"Visualization saved to: {output_dir / 'family_effectiveness.html'}")

## 5. ROI Calculation

In [ ]:
# Calculate ROI for preventive maintenance
# ROI = (Cost avoided - Cost invested) / Cost invested * 100

# Total preventive investment
total_preventive_cost = preventive['total_cost'].sum()

# Estimate cost avoided by preventive maintenance
# Assumption: Without preventive, all equipment would need corrective maintenance
# Calculate average corrective cost per equipment
avg_corrective_cost_per_equipment = corrective.groupby('code_equipement')['total_cost'].sum().mean()

# Number of unique equipment with preventive maintenance
equipment_with_preventive = preventive['code_equipement'].nunique()

# Estimated cost avoided (conservative: 60% of potential corrective costs)
AVOIDANCE_FACTOR = 0.6
estimated_cost_avoided = avg_corrective_cost_per_equipment * equipment_with_preventive * AVOIDANCE_FACTOR

# Calculate ROI
roi = ((estimated_cost_avoided - total_preventive_cost) / total_preventive_cost) * 100

print("\n=== ROI Analysis for Preventive Maintenance ===")
print(f"Total preventive maintenance investment: {total_preventive_cost:,.2f} EUR")
print(f"Estimated corrective costs avoided: {estimated_cost_avoided:,.2f} EUR")
print(f"Net benefit: {estimated_cost_avoided - total_preventive_cost:,.2f} EUR")
print(f"ROI: {roi:.2f}%")

if roi > 0:
    print(f"\nConclusion: Preventive maintenance has a POSITIVE ROI of {roi:.2f}%")
    print(f"For every 1 EUR invested in preventive maintenance, {1 + roi/100:.2f} EUR is returned.")
else:
    print(f"\nConclusion: Preventive maintenance has a NEGATIVE ROI of {roi:.2f}%")
    print("Consider optimizing preventive maintenance schedules to improve ROI.")

In [ ]:
# ROI by equipment family
roi_by_family = []

for family in family_comparison['famille_equipement'].unique():
    prev_cost = family_comparison[family_comparison['famille_equipement'] == family]['total_cost_preventive'].values[0]
    corr_cost = family_comparison[family_comparison['famille_equipement'] == family]['total_cost_corrective'].values[0]
    
    # Estimate avoided cost (60% of corrective costs)
    avoided = corr_cost * AVOIDANCE_FACTOR
    family_roi = ((avoided - prev_cost) / prev_cost) * 100 if prev_cost > 0 else 0
    
    roi_by_family.append({
        'famille_equipement': family,
        'preventive_cost': prev_cost,
        'corrective_cost': corr_cost,
        'estimated_avoided': avoided,
        'roi_percent': family_roi
    })

roi_df = pd.DataFrame(roi_by_family).sort_values('roi_percent', ascending=False)

print("\nROI by Equipment Family:")
print(roi_df.to_string(index=False))

roi_df.to_csv(output_dir / 'roi_by_family.csv', index=False)
print(f"\nROI analysis saved to: {output_dir / 'roi_by_family.csv'}")

In [ ]:
# Visualize ROI by family
fig = px.bar(
    roi_df.sort_values('roi_percent', ascending=True),
    x='roi_percent',
    y='famille_equipement',
    orientation='h',
    title='ROI of Preventive Maintenance by Equipment Family',
    labels={'roi_percent': 'ROI (%)', 'famille_equipement': 'Equipment Family'},
    color='roi_percent',
    color_continuous_scale='RdYlGn'
)

fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(height=600)
fig.show()

fig.write_html(output_dir / 'roi_by_family.html')
print(f"Visualization saved to: {output_dir / 'roi_by_family.html'}")

## 6. Temporal Effectiveness Analysis

In [ ]:
# Analyze effectiveness over time
corrective['year_month'] = corrective['date_creation_ot'].dt.to_period('M')
preventive['year_month'] = preventive['date_creation_ot'].dt.to_period('M')

corrective_temporal = corrective.groupby('year_month').agg({
    'code_ot': 'count',
    'total_cost': 'sum',
    'duration_hours': 'mean'
}).reset_index()
corrective_temporal.columns = ['year_month', 'n_interventions', 'total_cost', 'avg_duration']
corrective_temporal['maintenance_type'] = 'Corrective'

preventive_temporal = preventive.groupby('year_month').agg({
    'code_ot': 'count',
    'total_cost': 'sum',
    'duration_hours': 'mean'
}).reset_index()
preventive_temporal.columns = ['year_month', 'n_interventions', 'total_cost', 'avg_duration']
preventive_temporal['maintenance_type'] = 'Preventive'

temporal_combined = pd.concat([corrective_temporal, preventive_temporal])
temporal_combined['year_month_str'] = temporal_combined['year_month'].astype(str)

print("Temporal analysis completed")
print(f"Time range: {temporal_combined['year_month_str'].min()} to {temporal_combined['year_month_str'].max()}")

In [ ]:
# Visualize temporal trends
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Number of Interventions Over Time', 'Total Cost Over Time'),
    vertical_spacing=0.15
)

for mtype in ['Corrective', 'Preventive']:
    data = temporal_combined[temporal_combined['maintenance_type'] == mtype]
    
    fig.add_trace(
        go.Scatter(x=data['year_month_str'], y=data['n_interventions'],
                   mode='lines+markers', name=f'{mtype} - Count',
                   line=dict(width=2)),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=data['year_month_str'], y=data['total_cost'],
                   mode='lines+markers', name=f'{mtype} - Cost',
                   line=dict(width=2)),
        row=2, col=1
    )

fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_yaxes(title_text="Interventions", row=1, col=1)
fig.update_yaxes(title_text="Cost (EUR)", row=2, col=1)
fig.update_layout(height=800, title_text="Maintenance Trends Over Time")
fig.show()

fig.write_html(output_dir / 'temporal_effectiveness.html')
print(f"Visualization saved to: {output_dir / 'temporal_effectiveness.html'}")

## 7. Optimal Preventive Maintenance Schedule

In [ ]:
# Analyze time between preventive maintenance and subsequent corrective maintenance
# This helps identify optimal preventive maintenance intervals

equipment_with_both = set(corrective['code_equipement']).intersection(set(preventive['code_equipement']))
print(f"\nAnalyzing {len(equipment_with_both)} equipment with both maintenance types")

intervals = []

for equipment in equipment_with_both:
    prev_dates = preventive[preventive['code_equipement'] == equipment]['date_creation_ot'].sort_values()
    corr_dates = corrective[corrective['code_equipement'] == equipment]['date_creation_ot'].sort_values()
    
    for prev_date in prev_dates:
        # Find next corrective maintenance after this preventive
        next_corr = corr_dates[corr_dates > prev_date]
        if len(next_corr) > 0:
            days_until_failure = (next_corr.iloc[0] - prev_date).days
            intervals.append({
                'code_equipement': equipment,
                'preventive_date': prev_date,
                'corrective_date': next_corr.iloc[0],
                'days_until_failure': days_until_failure
            })

intervals_df = pd.DataFrame(intervals)

if len(intervals_df) > 0:
    print(f"\nAnalyzed {len(intervals_df)} preventive-to-corrective sequences")
    print(f"\nDays until failure after preventive maintenance:")
    print(intervals_df['days_until_failure'].describe())
    
    # Calculate optimal interval (median or 75th percentile)
    optimal_interval = intervals_df['days_until_failure'].quantile(0.75)
    print(f"\nRecommended preventive maintenance interval: {optimal_interval:.0f} days")
    print(f"This interval would prevent 75% of failures observed in the data.")
    
    intervals_df.to_csv(output_dir / 'maintenance_intervals.csv', index=False)
else:
    print("\nInsufficient data to calculate optimal intervals")

In [ ]:
# Visualize interval distribution
if len(intervals_df) > 0:
    fig = px.histogram(
        intervals_df,
        x='days_until_failure',
        nbins=30,
        title='Distribution of Days Until Failure After Preventive Maintenance',
        labels={'days_until_failure': 'Days Until Next Corrective Maintenance'},
        color_discrete_sequence=['#4ECDC4']
    )
    
    # Add vertical line for optimal interval
    fig.add_vline(x=optimal_interval, line_dash="dash", line_color="red",
                  annotation_text=f"Recommended: {optimal_interval:.0f} days")
    
    fig.show()
    fig.write_html(output_dir / 'optimal_intervals.html')
    print(f"Visualization saved to: {output_dir / 'optimal_intervals.html'}")

## 8. Predictive Model: Maintenance Effectiveness

In [ ]:
# Build a classifier to predict if preventive maintenance will be effective
# Effectiveness = no corrective maintenance needed within X days

EFFECTIVENESS_WINDOW = 90  # days

# Create training data from equipment with both maintenance types
effectiveness_data = []

for equipment in equipment_with_both:
    prev_records = preventive[preventive['code_equipement'] == equipment].sort_values('date_creation_ot')
    corr_records = corrective[corrective['code_equipement'] == equipment].sort_values('date_creation_ot')
    
    equipment_family = prev_records['famille_equipement'].iloc[0] if len(prev_records) > 0 else 'Unknown'
    
    for idx, prev_row in prev_records.iterrows():
        prev_date = prev_row['date_creation_ot']
        
        # Check if corrective maintenance occurred within effectiveness window
        next_corr = corr_records[corr_records['date_creation_ot'] > prev_date]
        
        if len(next_corr) > 0:
            days_to_next = (next_corr.iloc[0]['date_creation_ot'] - prev_date).days
            is_effective = days_to_next > EFFECTIVENESS_WINDOW
        else:
            # No corrective needed = effective
            is_effective = True
        
        effectiveness_data.append({
            'code_equipement': equipment,
            'famille_equipement': equipment_family,
            'prev_duration': prev_row['duration_hours'],
            'prev_cost': prev_row['total_cost'],
            'is_effective': is_effective
        })

effectiveness_df = pd.DataFrame(effectiveness_data)

print(f"\nCreated effectiveness dataset: {effectiveness_df.shape}")
print(f"Effective preventive maintenance: {effectiveness_df['is_effective'].sum()} ({effectiveness_df['is_effective'].mean()*100:.1f}%)")
print(f"Ineffective preventive maintenance: {(~effectiveness_df['is_effective']).sum()} ({(~effectiveness_df['is_effective']).mean()*100:.1f}%)")

In [ ]:
# Prepare features for modeling
if len(effectiveness_df) > 50:  # Need sufficient data
    # One-hot encode equipment family
    effectiveness_encoded = pd.get_dummies(effectiveness_df, columns=['famille_equipement'], prefix='family')
    
    # Prepare X and y
    feature_cols = [col for col in effectiveness_encoded.columns if col not in ['code_equipement', 'is_effective']]
    X = effectiveness_encoded[feature_cols]
    y = effectiveness_encoded['is_effective'].astype(int)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"\nTraining set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")
    print(f"Features: {len(feature_cols)}")
else:
    print("\nInsufficient data for predictive modeling")
    X_train = None

In [ ]:
# Train Random Forest classifier
if X_train is not None:
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    
    rf_model.fit(X_train, y_train)
    
    # Predictions
    y_pred = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
    
    # Evaluation
    print("\n=== Random Forest Model Performance ===")
    print(classification_report(y_test, y_pred, target_names=['Ineffective', 'Effective']))
    
    # ROC AUC
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"\nROC AUC Score: {roc_auc:.3f}")
    
    # Cross-validation
    cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='roc_auc')
    print(f"Cross-validation ROC AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

In [ ]:
# Feature importance
if X_train is not None:
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10).to_string(index=False))
    
    # Visualize feature importance
    fig = px.bar(
        feature_importance.head(15),
        x='importance',
        y='feature',
        orientation='h',
        title='Top 15 Features for Predicting Maintenance Effectiveness',
        labels={'importance': 'Importance Score', 'feature': 'Feature'}
    )
    fig.update_layout(height=600)
    fig.show()
    
    fig.write_html(output_dir / 'effectiveness_feature_importance.html')
    print(f"\nFeature importance visualization saved to: {output_dir / 'effectiveness_feature_importance.html'}")

In [ ]:
# Confusion matrix
if X_train is not None:
    cm = confusion_matrix(y_test, y_pred)
    
    fig = px.imshow(
        cm,
        labels=dict(x="Predicted", y="Actual", color="Count"),
        x=['Ineffective', 'Effective'],
        y=['Ineffective', 'Effective'],
        text_auto=True,
        title='Confusion Matrix - Maintenance Effectiveness Prediction',
        color_continuous_scale='Blues'
    )
    fig.show()
    
    fig.write_html(output_dir / 'effectiveness_confusion_matrix.html')
    print(f"Confusion matrix saved to: {output_dir / 'effectiveness_confusion_matrix.html'}")

## 9. Save Models

In [ ]:
# Save the effectiveness prediction model
if X_train is not None:
    with open(models_dir / 'maintenance_effectiveness_model.pkl', 'wb') as f:
        pickle.dump(rf_model, f)
    
    # Save feature columns for future predictions
    with open(models_dir / 'effectiveness_features.pkl', 'wb') as f:
        pickle.dump(feature_cols, f)
    
    print(f"\nModels saved to: {models_dir}")
    print(f"- maintenance_effectiveness_model.pkl")
    print(f"- effectiveness_features.pkl")
else:
    print("\nNo model trained due to insufficient data")

## 10. Key Findings and Recommendations

In [ ]:
# Generate comprehensive summary report
summary_report = {
    'analysis_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'overall_metrics': {
        'total_corrective_interventions': len(corrective),
        'total_preventive_interventions': len(preventive),
        'total_corrective_cost_eur': float(corrective['total_cost'].sum()),
        'total_preventive_cost_eur': float(preventive['total_cost'].sum()),
        'avg_corrective_cost_eur': float(corrective['total_cost'].mean()),
        'avg_preventive_cost_eur': float(preventive['total_cost'].mean()),
        'avg_corrective_duration_hours': float(corrective['duration_hours'].mean()),
        'avg_preventive_duration_hours': float(preventive['duration_hours'].mean())
    },
    'roi_analysis': {
        'preventive_investment_eur': float(total_preventive_cost),
        'estimated_cost_avoided_eur': float(estimated_cost_avoided),
        'net_benefit_eur': float(estimated_cost_avoided - total_preventive_cost),
        'roi_percent': float(roi)
    },
    'top_performing_families': family_comparison.nlargest(3, 'effectiveness_score')[['famille_equipement', 'effectiveness_score']].to_dict('records'),
    'lowest_performing_families': family_comparison.nsmallest(3, 'effectiveness_score')[['famille_equipement', 'effectiveness_score']].to_dict('records')
}

if len(intervals_df) > 0:
    summary_report['optimal_interval_days'] = float(optimal_interval)

if X_train is not None:
    summary_report['model_performance'] = {
        'roc_auc': float(roc_auc),
        'cv_roc_auc_mean': float(cv_scores.mean()),
        'cv_roc_auc_std': float(cv_scores.std())
    }

print("\n" + "="*80)
print("MAINTENANCE EFFECTIVENESS ANALYSIS - SUMMARY REPORT")
print("="*80)

print("\n1. OVERALL METRICS")
print(f"   - Corrective interventions: {summary_report['overall_metrics']['total_corrective_interventions']}")
print(f"   - Preventive interventions: {summary_report['overall_metrics']['total_preventive_interventions']}")
print(f"   - Total corrective cost: {summary_report['overall_metrics']['total_corrective_cost_eur']:,.2f} EUR")
print(f"   - Total preventive cost: {summary_report['overall_metrics']['total_preventive_cost_eur']:,.2f} EUR")

print("\n2. ROI ANALYSIS")
print(f"   - Preventive maintenance investment: {summary_report['roi_analysis']['preventive_investment_eur']:,.2f} EUR")
print(f"   - Estimated costs avoided: {summary_report['roi_analysis']['estimated_cost_avoided_eur']:,.2f} EUR")
print(f"   - Net benefit: {summary_report['roi_analysis']['net_benefit_eur']:,.2f} EUR")
print(f"   - ROI: {summary_report['roi_analysis']['roi_percent']:.2f}%")

print("\n3. EQUIPMENT FAMILY PERFORMANCE")
print("   Top 3 performers:")
for family in summary_report['top_performing_families']:
    print(f"   - {family['famille_equipement']}: {family['effectiveness_score']:.1f}/100")
print("   Bottom 3 performers:")
for family in summary_report['lowest_performing_families']:
    print(f"   - {family['famille_equipement']}: {family['effectiveness_score']:.1f}/100")

if 'optimal_interval_days' in summary_report:
    print("\n4. OPTIMAL MAINTENANCE SCHEDULE")
    print(f"   - Recommended preventive interval: {summary_report['optimal_interval_days']:.0f} days")

if 'model_performance' in summary_report:
    print("\n5. PREDICTIVE MODEL PERFORMANCE")
    print(f"   - ROC AUC Score: {summary_report['model_performance']['roc_auc']:.3f}")
    print(f"   - Cross-validation ROC AUC: {summary_report['model_performance']['cv_roc_auc_mean']:.3f} (+/- {summary_report['model_performance']['cv_roc_auc_std']:.3f})")

print("\n6. KEY RECOMMENDATIONS")
recommendations = []

if roi > 0:
    recommendations.append("Continue and expand preventive maintenance programs (positive ROI)")
else:
    recommendations.append("Optimize preventive maintenance schedules to improve ROI")

if len(intervals_df) > 0:
    recommendations.append(f"Implement preventive maintenance every {optimal_interval:.0f} days")

# Check for families with low effectiveness
low_performers = family_comparison[family_comparison['effectiveness_score'] < 30]
if len(low_performers) > 0:
    recommendations.append(f"Focus improvement efforts on {len(low_performers)} low-performing equipment families")

# Check cost ratio
if corrective['total_cost'].mean() > preventive['total_cost'].mean() * 2:
    recommendations.append("Increase preventive maintenance frequency to reduce high corrective costs")

for i, rec in enumerate(recommendations, 1):
    print(f"   {i}. {rec}")

print("\n" + "="*80)

# Save summary report
import json
with open(output_dir / 'maintenance_effectiveness_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print(f"\nSummary report saved to: {output_dir / 'maintenance_effectiveness_summary.json'}")

## 11. Outputs Summary

In [ ]:
print("\n" + "="*80)
print("PHASE 4: MAINTENANCE EFFECTIVENESS EVALUATION - COMPLETE")
print("="*80)

print("\nFiles Generated:")
print("\nCSV Reports:")
print(f"  - {output_dir / 'maintenance_comparison.csv'}")
print(f"  - {output_dir / 'family_effectiveness.csv'}")
print(f"  - {output_dir / 'roi_by_family.csv'}")
if len(intervals_df) > 0:
    print(f"  - {output_dir / 'maintenance_intervals.csv'}")

print("\nVisualizations (HTML):")
print(f"  - {output_dir / 'maintenance_comparison.html'}")
print(f"  - {output_dir / 'family_effectiveness.html'}")
print(f"  - {output_dir / 'roi_by_family.html'}")
print(f"  - {output_dir / 'temporal_effectiveness.html'}")
if len(intervals_df) > 0:
    print(f"  - {output_dir / 'optimal_intervals.html'}")
if X_train is not None:
    print(f"  - {output_dir / 'effectiveness_feature_importance.html'}")
    print(f"  - {output_dir / 'effectiveness_confusion_matrix.html'}")

if X_train is not None:
    print("\nTrained Models:")
    print(f"  - {models_dir / 'maintenance_effectiveness_model.pkl'}")
    print(f"  - {models_dir / 'effectiveness_features.pkl'}")

print("\nSummary Report:")
print(f"  - {output_dir / 'maintenance_effectiveness_summary.json'}")

print("\n" + "="*80)
print("This analysis directly addresses the requirement:")
print('"Évaluer l\'efficacité des deux types de maintenance"')
print("="*80)

print("\nNext steps:")
print("  1. Create phase4_spare_parts_forecasting.ipynb")
print("  2. Create phase4_model_evaluation.ipynb")
print("  3. Update Streamlit dashboard with predictions")